# Intelligent Prompt Optimization Assistant: NLP Preprocessing Pipeline
### Major Project — Phase 2
**Objective:** Build, validate, and serialize the Natural Language Processing (NLP) pipeline for converting raw user prompts into clean, uniform numerical token sequences ready for Deep Learning (1D-CNN and Bi-LSTM) architectures.

**Key Pipeline Stages:**
1. Dataset Ingestion & Missing Value Auditing
2. Exploratory Data Analysis (EDA) on Text Lengths & Word Frequencies
3. Text Normalization: Contraction Expansion, Lowercasing & Punctuation Cleaning
4. Stratified Train / Validation / Test Splitting (70% / 15% / 15%)
5. Vocabulary Building & Sequence Tokenization (**Fitted on Training Set Only to Prevent Data Leakage**)
6. Sequence Padding Analysis (`max_len = 120` tokens)
7. Categorical Target Label Encoding (`Poor: 0, Average: 1, Good: 2`)
8. Serialization of Preprocessing Artifacts (`tokenizer.pkl`, `label_encoder.pkl`)

In [1]:
# Step 1: Environment & Imports
import os
import sys
import csv
import json
import pickle
import re
from collections import Counter

# Add project root to path for local or Colab execution
PROJECT_ROOT = os.path.dirname(os.path.abspath(""))
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

print(f"[*] Project Root Directory: {PROJECT_ROOT}")

[*] Project Root Directory: c:\Users\Sagar\.gemini\antigravity\scratch\Intelligent-Prompt-Optimizer


In [2]:
# Step 2: Load the Raw Dataset (dataset/prompts.csv)
DATASET_PATH = os.path.join(PROJECT_ROOT, "dataset", "prompts.csv")

records = []
with open(DATASET_PATH, mode="r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        records.append(row)

print(f"[+] Ingested {len(records)} total prompt samples across 6 domains and 3 quality tiers.")
print(f"[+] Sample Columns: {list(records[0].keys())}")

[+] Ingested 1260 total prompt samples across 6 domains and 3 quality tiers.
[+] Sample Columns: ['prompt_id', 'prompt', 'category', 'clarity_score', 'context_score', 'specificity_score', 'completeness_score', 'overall_score', 'quality_label', 'weaknesses', 'optimized_prompt']


In [3]:
# Step 3: Text Cleaning and Normalization
from utils.preprocessing import clean_text

for r in records:
    r["clean_prompt"] = clean_text(r["prompt"])

print("[*] Sample Text Before vs After Cleaning:")
print(f"    Original: {records[0]['prompt'][:80]}...")
print(f"    Cleaned : {records[0]['clean_prompt'][:80]}...")

[*] Sample Text Before vs After Cleaning:
    Original: Create an executive summary for Employee return-to-office transition plan. Targe...
    Cleaned : create an executive summary for employee return to office transition plan. targe...


In [4]:
# Step 4: Stratified Splitting (70% Train, 15% Validation, 15% Test)
from utils.preprocessing import stratified_split

train_set, val_set, test_set = stratified_split(
    records, train_ratio=0.70, val_ratio=0.15, test_ratio=0.15, seed=42
)

print(f"[+] Train Set Samples      : {len(train_set)} (70%)")
print(f"[+] Validation Set Samples : {len(val_set)} (15%)")
print(f"[+] Test Set Samples       : {len(test_set)} (15%)")

[+] Train Set Samples      : 882 (70%)
[+] Validation Set Samples : 189 (15%)
[+] Test Set Samples       : 189 (15%)


In [5]:
# Step 5: Fit Tokenizer on Training Data ONLY (Zero Data Leakage)
from utils.preprocessing import TextTokenizer, QualityLabelEncoder, DEFAULT_MAX_LEN

train_texts = [r["clean_prompt"] for r in train_set]
tokenizer = TextTokenizer(max_words=5000)
tokenizer.fit_on_texts(train_texts)

print(f"[+] Fitted Vocabulary Size: {tokenizer.vocab_size} tokens")
print(f"[+] Top 10 Most Common Words: {tokenizer.word_counts.most_common(10)}")

[+] Fitted Vocabulary Size: 1343 tokens
[+] Top 10 Most Common Words: [('and', 1586), ('a', 879), ('for', 524), ('an', 521), ('the', 518), ('with', 470), ('to', 430), ('&', 394), ('in', 342), ('of', 322)]


In [6]:
# Step 6: Convert to Uniform Padded Sequences (max_len = 120)
X_train = tokenizer.pad_sequences(tokenizer.texts_to_sequences(train_texts), maxlen=DEFAULT_MAX_LEN)
X_val = tokenizer.pad_sequences(tokenizer.texts_to_sequences([r["clean_prompt"] for r in val_set]), maxlen=DEFAULT_MAX_LEN)
X_test = tokenizer.pad_sequences(tokenizer.texts_to_sequences([r["clean_prompt"] for r in test_set]), maxlen=DEFAULT_MAX_LEN)

label_encoder = QualityLabelEncoder()
y_train = label_encoder.transform([r["quality_label"] for r in train_set])
y_val = label_encoder.transform([r["quality_label"] for r in val_set])
y_test = label_encoder.transform([r["quality_label"] for r in test_set])

print(f"[+] X_train Matrix Shape: ({len(X_train)}, {len(X_train[0])})")
print(f"[+] X_val Matrix Shape  : ({len(X_val)}, {len(X_val[0])})")
print(f"[+] X_test Matrix Shape : ({len(X_test)}, {len(X_test[0])})")
print(f"[+] Encoded Classes     : {label_encoder.classes_}")

[+] X_train Matrix Shape: (882, 120)
[+] X_val Matrix Shape  : (189, 120)
[+] X_test Matrix Shape : (189, 120)
[+] Encoded Classes     : ['Poor', 'Average', 'Good']


### Phase 2 Summary Checkpoint
The dataset is cleaned, tokenized, padded to 120 tokens, and split into zero-leakage training, validation, and test subsets. These artifacts are stored in `models/` and `dataset/preprocessed_splits.json` and are ready for **Phase 3 (1D-CNN Training)** and **Phase 4 (Bi-LSTM Training)**.